# Analyse d'un asservissement SISO : NumPy / SciPy, python-control, minilink

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/courses/udes_gro501/siso_transfer_function_analysis.ipynb)

Les commandes de base pour analyser un asservissement à une entrée et une sortie, sur un système masse–ressort–amortisseur :

1. **Système** : la fonction de transfert $H(s) = y/u$.
2. **Boucle ouverte** : réponse à un échelon, pôles et zéros, réponse fréquentielle.
3. **Boucle fermée** : un contrôleur $C(s) = u/e$, le chemin direct $L(s) = C(s)H(s)$ (lieu des racines, Bode), puis $CL(s) = y/r$ (échelon, pôles et zéros).

La même séquence trois fois : avec **NumPy / SciPy / Matplotlib** seulement, avec **python-control**, puis avec **minilink**. Quand une bibliothèque n'a pas l'outil, l'étape est simplement sautée.

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

## Partie 1 — NumPy, SciPy et Matplotlib

Pas de variable $s$ : une fonction de transfert est une liste de coefficients au numérateur et une au dénominateur, par puissances décroissantes de $s$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

### Système

$H(s) = \dfrac{y}{u} = \dfrac{1}{m s^2 + b s + k}$

In [ ]:
m, b, k = 10.0, 2.0, 10.0

num_H, den_H = [1.0], [m, b, k]
H = signal.TransferFunction(num_H, den_H)
H  # SciPy normalise le dénominateur

### Analyse de la boucle ouverte

Réponse temporelle à un échelon $u = 1$ (le transitoire), position des pôles et des zéros, puis réponse fréquentielle à $u = \sin(\omega t)$ (le régime permanent).

In [ ]:
t, y = signal.step(H)

plt.plot(t, y)
plt.xlabel("t [s]")
plt.ylabel("y")
plt.grid(True)
plt.show()

In [ ]:
plt.plot(H.poles.real, H.poles.imag, "x", label="pôles")
plt.plot(H.zeros.real, H.zeros.imag, "o", label="zéros")
plt.axis("equal")
plt.xlabel("Re")
plt.ylabel("Im")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
w, mag, phase = signal.bode(H)

fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].semilogx(w, mag)
ax[1].semilogx(w, phase)
ax[0].set_ylabel("|H| [dB]")
ax[1].set_ylabel("phase [°]")
ax[1].set_xlabel("ω [rad/s]")
ax[0].grid(True, which="both")
ax[1].grid(True, which="both")
plt.show()

### Analyse de la boucle fermée

Contrôleur PID, $C(s) = \dfrac{u}{e} = k_p + k_d s + \dfrac{k_i}{s} = \dfrac{k_d s^2 + k_p s + k_i}{s}$, et chemin direct $L(s) = \dfrac{y}{e} = C(s)H(s)$.

SciPy n'a pas d'algèbre sur les fonctions de transfert : le produit $C H$ se fait sur les polynômes avec `np.polymul`.

In [ ]:
kp, kd, ki = 1.0, 1.0, 1.0

num_C, den_C = [kd, kp, ki], [1.0, 0.0]
num_L, den_L = np.polymul(num_C, num_H), np.polymul(den_C, den_H)
L = signal.TransferFunction(num_L, den_L)
L

Lieu des racines : pas d'outil dans SciPy, étape sautée.

In [ ]:
w, mag, phase = signal.bode(L)

fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].semilogx(w, mag)
ax[1].semilogx(w, phase)
ax[0].set_ylabel("|L| [dB]")
ax[1].set_ylabel("phase [°]")
ax[1].set_xlabel("ω [rad/s]")
ax[0].grid(True, which="both")
ax[1].grid(True, which="both")
plt.show()

Fonction de transfert en boucle fermée, $CL(s) = \dfrac{y}{r} = \dfrac{L}{1 + L} = \dfrac{N_L}{D_L + N_L}$ : une somme de polynômes avec `np.polyadd`, déjà sous forme minimale.

In [ ]:
CL = signal.TransferFunction(num_L, np.polyadd(den_L, num_L))
CL

In [ ]:
t, y = signal.step(CL)

plt.plot(t, y)
plt.xlabel("t [s]")
plt.ylabel("y")
plt.grid(True)
plt.show()

In [ ]:
plt.plot(CL.poles.real, CL.poles.imag, "x", label="pôles")
plt.plot(CL.zeros.real, CL.zeros.imag, "o", label="zéros")
plt.axis("equal")
plt.xlabel("Re")
plt.ylabel("Im")
plt.legend()
plt.grid(True)
plt.show()

## Partie 2 — python-control

La bibliothèque [python-control](https://python-control.readthedocs.io/en/0.10.2/) reprend les commandes de MATLAB : une variable $s$, l'algèbre des fonctions de transfert et les tracés en une ligne.

In [ ]:
import importlib.util

if importlib.util.find_spec("control") is None:
    get_ipython().system("pip install -q control")

In [ ]:
import control as ct

### Système

In [ ]:
s = ct.tf("s")

m, b, k = 10.0, 2.0, 10.0

H = 1 / (m * s**2 + b * s + k)
H

### Analyse de la boucle ouverte

In [ ]:
ct.step_response(H).plot();

In [ ]:
ct.pzmap(H);

In [ ]:
ct.bode_plot(H);

### Analyse de la boucle fermée

In [ ]:
kp, kd, ki = 1.0, 1.0, 1.0

C = kp + kd * s + ki / s
L = C * H
L

In [ ]:
ct.root_locus(L);

In [ ]:
ct.bode_plot(L);

`minreal` simplifie les facteurs communs que l'algèbre $L / (1 + L)$ duplique.

In [ ]:
CL = ct.minreal(L / (1 + L), verbose=False)
CL

In [ ]:
ct.step_response(CL).plot();

In [ ]:
ct.pzmap(CL);

## Partie 3 — minilink

Dans [minilink](https://github.com/alx87grd/minilink), une fonction de transfert est un bloc : `>>` met deux blocs en série, `@` ferme la boucle avec $e = r - y$, et chaque outil d'analyse est un verbe `plot_…` du bloc. Pas de variable $s$ ni de `minreal` : ces deux étapes sont sautées.

In [ ]:
from minilink import PID, TransferFunction

### Système

In [ ]:
m, b, k = 10.0, 2.0, 10.0

H = TransferFunction([1.0], [m, b, k])

### Analyse de la boucle ouverte

In [ ]:
H.plot_step_response()

In [ ]:
H.plot_pzmap()

In [ ]:
H.plot_bode(margins=False)

### Analyse de la boucle fermée

Le `PID` de minilink filtre sa dérivée, $k_d \dfrac{s}{\tau s + 1}$ : un pôle rapide de plus en $-1/\tau = -100$ ($\tau = 0{,}01$ s), visible loin à gauche sur le lieu des racines et sur la carte des pôles. On zoome près de l'origine.

In [ ]:
kp, kd, ki = 1.0, 1.0, 1.0

C = PID(Kp=kp, Ki=ki, Kd=kd, tau=0.01)
L = C >> H

In [ ]:
lieu = L.plot_root_locus(show=False)
lieu.axes.set_xlim(-1.5, 0.5)
lieu.axes.set_ylim(-1.5, 1.5);

In [ ]:
L.plot_bode()

In [ ]:
CL = C @ H
CL.plot_diagram()

In [ ]:
CL.plot_step_response()

In [ ]:
carte = CL.plot_pzmap(show=False)
carte.axes.set_xlim(-1.5, 0.5)
carte.axes.set_ylim(-1.5, 1.5);